In [17]:
import json
import pandas as pd
import time
import datetime
import orjson

In [18]:
start_time_full = time.time()
start_time_full_str = datetime.datetime.now().strftime("%H:%M:%S")

In [19]:
with open('../raw_data/events.json', 'r') as f:
    data = json.load(f)

In [20]:
start_time = time.time()
start_time_str = datetime.datetime.now().strftime("%H:%M:%S")

In [21]:
data_df = pd.json_normalize(data,sep='_')

In [22]:
params_data =[
    {item['key']:next(v for v in item['value'].values() if v is not None)
     for item in params_list}
    for params_list in data_df['event_params']
]

In [23]:
param_df = pd.DataFrame(params_data).add_prefix('ep_')

In [24]:
user_props_data =[
    {
        prop['key']:next(v for k, v in prop['value'].items()if v is not None and k!='set_timestamp_micros')
        for prop in props_list
    }
    for props_list in data_df['user_properties']
]

In [25]:
user_props_df = pd.DataFrame(user_props_data).add_prefix('user_prop_')


In [26]:
event_date_dt = pd.to_datetime(data_df["event_date"], format="%Y%m%d")
data_df['year']= event_date_dt.dt.year
data_df['month'] = event_date_dt.dt.month

In [27]:
final_df = pd.concat([data_df, param_df, user_props_df], axis=1)
final_df.drop(columns=['user_properties', 'event_params'], inplace=True)

In [28]:
end_time = time.time()
end_time_str = datetime.datetime.now().strftime("%H:%M:%S")
print(end_time_str)
total_time = end_time - start_time
print(f"Total processing time without data loading and exporting: {total_time:.2f} seconds")

17:47:54
Total processing time without data loading and exporting: 53.01 seconds


In [29]:
final_df.to_csv("../CSV_output/events_normalized.csv", index=False)

In [30]:
total_end_time = time.time()
total_end_time_str = datetime.datetime.now().strftime("%H:%M:%S")
total_time = total_end_time - start_time_full
print(f"Total processing time: {total_time:.2f} seconds")

Total processing time: 101.48 seconds


In [31]:
print(76.5/60)

1.275


In [32]:
import pandas as pd

In [40]:
data = pd.read_csv("../CSV_output/events_normalized.csv", low_memory=False)


In [41]:
cols = data.columns.tolist()

ep_cols = [c for c in cols if c.startswith("ep_")]

before = []
after = []
passed_event_name = False

for c in cols:
    if c == "event_name":
        passed_event_name = True
        before.append(c)
    elif c.startswith("ep_"):
        continue
    else:
        if not passed_event_name:
            before.append(c)
        else:
            after.append(c)

new_order = before + ep_cols + after

data = data[new_order]

In [42]:
data.to_parquet("../parquet_output/events_normalized.parquet", engine="pyarrow", index=False)


In [43]:
data = pd.read_parquet("../parquet_output/events_normalized.parquet")

In [44]:
data

,event_date,event_timestamp,event_name,ep_batch_page_id,ep_source,ep_page_location,ep_engaged_session_event,ep_campaign,ep_batch_ordering_id,ep_page_title,...,session_traffic_source_last_click_google_ads_campaign_campaign_id,session_traffic_source_last_click_google_ads_campaign_campaign_name,session_traffic_source_last_click_google_ads_campaign_ad_group_id,session_traffic_source_last_click_google_ads_campaign_ad_group_name,user_ltv_revenue,user_ltv_currency,year,month,user_prop_user_session_id,user_prop_user_client_id
0,20241113,1731513971041603,first_visit,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
1,20241113,1731513971041603,session_start,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
2,20241113,1731513971041603,page_view,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,None,None
3,20241113,1731513976070304,user_session_info,1731513969906,None,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,None,2,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,_1731513970,_2091574202.1731513971
4,20241113,1731513976070304,scroll,1731513969906,None,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,None,2,Eye Care Center in Crystal Lake - Shopko Optical,...,None,None,None,None,NaN,None,2024,11,_1731513970,_2091574202.1731513971
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260860,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260861,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
260862,20241113,1731552192524186,scroll,1731552185721,None,https://www.shopko.com/eye-care/?utm_source=em...,1.0,None,2,Eye Exam Center Locations Near You,...,None,None,None,None,NaN,None,2024,11,_1731552186,_1436230623.1728949201
